<a href="https://colab.research.google.com/github/acac2495/2D-BCH-Codes/blob/main/AllBCH.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [50]:
import numpy as np
import argparse
import itertools
import random

from tqdm.notebook import tqdm

In [30]:
#Basic building block functions

ALPHA_POWERS = [8, 4, 2, 1, 12, 6, 3, 13, 10, 5, 14, 7, 15, 11, 9]  # a^0..a^14, MSB-first

EXP_STR = {0: "0"}
for idx, val in enumerate(ALPHA_POWERS):
    EXP_STR[val] = f"a^{idx}"

def alpha_pow(exp, n=15):
    return ALPHA_POWERS[exp % n]

def to_hex(val):
    return format(val, 'X')

print(EXP_STR)

{0: '0', 8: 'a^0', 4: 'a^1', 2: 'a^2', 1: 'a^3', 12: 'a^4', 6: 'a^5', 3: 'a^6', 13: 'a^7', 10: 'a^8', 5: 'a^9', 14: 'a^10', 7: 'a^11', 15: 'a^12', 11: 'a^13', 9: 'a^14'}


In [3]:
N = 15
t = 2
m = 4

In [4]:
#Galois field operations

def gf_mul(a, b):
    if(a == 0 or b == 0):
        return 0
    p1 = ALPHA_POWERS.index(a)
    p2 = ALPHA_POWERS.index(b)
    return ALPHA_POWERS[(p1 + p2) % N]

def gf_pow(a, pow):
    if(a == 0):
        return 8
    p1 = ALPHA_POWERS.index(a)
    return ALPHA_POWERS[(p1 * pow) % N]

def gf_inv(a):
    if(a == 0):
        return 0
    p1 = ALPHA_POWERS.index(a)
    return ALPHA_POWERS[(N - p1) % N]

def alpha_pow(exp):
    return ALPHA_POWERS[exp % N]

In [37]:
#Discrete Finite Field Fourier Transform

def dffft_pt(c, j, jp):
    acc = 0
    for i in range(N):
        for ip in range(N):
            if(c[i][ip]):
                acc ^= alpha_pow(i * j + ip * jp)
    return acc

def dffft_2t(c):
    C = [[0] * (2*t) for _ in range(2*t)]
    for j in range(1, 2*t + 1):
        for jp in range(1, 2*t + 1):
            C[j-1][jp-1] = dffft_pt(c, j, jp)
    return C

def dffft_full(c):
    C = [[0] * N for _ in range(N)]
    for j in range(1, N + 1):
        for jp in range(1, N + 1):
            C[j-1][jp-1] = dffft_pt(c, j, jp)
    return C

def print_grid(C):
    for row in C:
        for elem in row:
            print(EXP_STR[elem], " | " , end=" ")
        print("")

def print_bin(err):
    for row in err:
        print(row)

In [35]:
#Inverse Discrete Finite Field Fourier Transform

def idffft_pt(C, i, ip):
    acc = 0
    for j in range(N):
        for jp in range(N):
            p1 = gf_inv(alpha_pow(i * j + ip * jp))
            acc ^= gf_mul(p1, C[j][jp])
    if(acc != 0):
        return 1
    else:
        return 0

def idffft_full(C):
    c = [[0] * N for _ in range(N)]
    for i in range(N):
        for ip in range(N):
            c[i][ip] = idffft_pt(C, i, ip)
    return c


In [6]:
#DFFFT Test

c = [[0] * N for _ in range(N)]
c[5][5] = 1
c[5][6] = 1
c[6][5] = 1
c[6][6] = 1

C = dffft_2t(c)
print_grid(C)

print(" ")

C_arrnp = np.array(C)
C_colwise = np.transpose(C_arrnp).tolist()
print_grid(C_colwise)

a^3 a^12 a^8 a^0 
a^12 a^6 a^2 a^9 
a^8 a^2 a^13 a^5 
a^0 a^9 a^5 a^12 
 
a^3 a^12 a^8 a^0 
a^12 a^6 a^2 a^9 
a^8 a^2 a^13 a^5 
a^0 a^9 a^5 a^12 


In [7]:
#Berlekamp Massey Engine

def BM(synds):
    sigma = [0] * (t + 1)
    sigma_p = [0] * (t + 1)
    d_p = 8
    for i in range (0, t + 1):
        if(i == 0):
            sigma[i] = 8
        if(i == 1):
            sigma_p[i] = 8

    temp_reg = [0] * (t + 1)
    for i in range(2 * t):
        s_in = synds[i]
        temp_reg = [s_in] + temp_reg[0:t]

        d_mu = temp_reg[0]
        for j in range(1, t+1):
            d_mu ^= gf_mul(temp_reg[j], sigma[j])

        dmu_dpinv = gf_mul(d_mu, gf_inv(d_p))
        corr = [gf_mul(sigma_p[i], dmu_dpinv) for i in range(t+1)]

        sigma_next = [sigma[j] ^ corr[j] for j in range(t+1)]
        load_sigma = 1 if d_mu != 0 else 0
        sigma_store = sigma[:] if load_sigma else sigma_p[:]

        sigma_p = [0] + sigma_store[0:t]
        sigma = sigma_next

        if(load_sigma):
            d_p = d_mu

    return sigma

In [45]:
#Fixed Berlekamp Massey Engine

# Berlekamp-Massey engine, register-style like your BM(), with the length register added.
# This is the same algorithm the patched BM_top.v should implement.
def BM_fixed(synds):
    sigma   = [0] * (t + 1);  sigma[0] = 8            # sigma(x) = 1
    sigma_p = [0] * (t + 1);  sigma_p[1] = 8          # x * sigma_p(x) kept pre-shifted
    d_p = 8
    L = 0                                             # length register
    temp_reg = [0] * (t + 1)
    for i in range(2 * t):                            # i is the iteration number (= count in the RTL)
        temp_reg = [synds[i]] + temp_reg[0:t]
        d_mu = temp_reg[0]
        for j in range(1, t + 1):
            d_mu ^= gf_mul(temp_reg[j], sigma[j])

        dmu_dpinv = gf_mul(d_mu, gf_inv(d_p))
        corr = [gf_mul(sigma_p[j], dmu_dpinv) for j in range(t + 1)]
        sigma_next = [sigma[j] ^ corr[j] for j in range(t + 1)]    # still corrected on EVERY nonzero d

        load_sigma = 1 if (d_mu != 0 and 2 * L <= i) else 0        # only a LENGTH CHANGE reloads the backup
        sigma_store = sigma[:] if load_sigma else sigma_p[:]
        sigma_p = [0] + sigma_store[0:t]
        sigma = sigma_next
        if load_sigma:
            d_p = d_mu
            L = i + 1 - L
    return sigma

In [23]:
#Polynomial and Root functions

def print_poly(sigma):
    powers = []
    for coeff in sigma:
        powers.append(EXP_STR[coeff])
    print(powers)

def print_roots(sigma):
    for i in ALPHA_POWERS:
        res = 8
        for j in range(1, t+1):
            pow = gf_pow(i, j)
            res ^= gf_mul(pow, sigma[j])
        if(res == 0):
            print(EXP_STR[i], end=" ")

def display_roots(roots):
    for root in roots:
        print(EXP_STR[root], end=" ")
    print("")

def get_rpos(sigma):
    rpos = []
    for i in ALPHA_POWERS:
        res = 8
        for j in range(1, t+1):
            pow = gf_pow(i, j)
            res ^= gf_mul(pow, sigma[j])
        if(res == 0):
            rpos.append(1)
        else:
            rpos.append(0)
    return rpos

def root_lookup(rpos):
    count = 0
    roots = []
    for i in range(len(rpos)):
        if(rpos[i] == 1):
            roots.append(ALPHA_POWERS[i])
            count += 1
        if(count == t):
            break
    return roots

def root_to_coeff(roots):
    if len(roots) >= 2:
        a1, a2 = roots[0], roots[1]
        prod = gf_mul(a1, a2)
        return [8, gf_mul(a1 ^ a2, gf_inv(prod)), gf_inv(prod)]
    if len(roots) == 1:
        return [8, gf_inv(roots[0]), 0]
    return [8, 0, 0]

In [25]:
#Recursive Extension basic functions

def RE_base(c, window):
    res = 0
    for i in range(1, t + 1):
        res ^= gf_mul(c[i], window[-i])
    return res

def RE_chain(c, S_init):
    window = list(S_init)
    out = []
    for _ in range(N - 2 * t):
        new = RE_base(c, window)
        out.append(new)
        window = window[1:] + [new]
    return out


In [24]:
#Recursive extension row and column wise

def extend_columns(S_2t, col_ccp_poly):
    S_full = [[0] * (2 * t) for _ in range(N)]
    for k in range(2*t):
        seq = [S_2t[j][k] for j in range(2*t)]
        seq_full = seq + RE_chain(col_ccp_poly, seq[-t:])
        for j in range(N):
            S_full[j][k] = seq_full[j]
    return S_full

def extend_rows(col_full, row_ccp_poly):
    S_full = []
    for j in range(N):
        seq = list(col_full[j])
        S_full.append(seq + RE_chain(row_ccp_poly, seq[-t:]))
    return S_full

In [49]:
#Full datapath

def ecc_2d(r_in):
    print("Input : ")
    print_bin(r_in)
    print("")

    S_2t = dffft_2t(r_in)
    S_2t_arrnp = np.array(S_2t)
    S_2t_colwise = np.transpose(S_2t_arrnp).tolist()

    print("Syndromes 1-2t : ")
    print_grid(S_2t)

    print("")

    col_poly = []
    row_poly = []

    for j in S_2t:
        row_poly.append(BM_fixed(j))
    for j in S_2t_colwise:
        col_poly.append(BM_fixed(j))

    print("Column sigmas : ")
    for sigma in col_poly:
        print_poly(sigma)

    print("Row sigmas : ")
    for sigma in row_poly:
        print_poly(sigma)

    col_rpos = []
    row_rpos = []

    for sigma in col_poly:
        col_rpos.append(get_rpos(sigma))
    for sigma in row_poly:
        row_rpos.append(get_rpos(sigma))

    col_rpos_lcm = 0
    row_rpos_lcm = 0

    col_rpos_lcm = [0] * N
    row_rpos_lcm = [0] * N

    for rpos in col_rpos:
        col_rpos_lcm = [a | b for a, b in zip(col_rpos_lcm, rpos)]
    for rpos in row_rpos:
        row_rpos_lcm = [a | b for a, b in zip(row_rpos_lcm, rpos)]

    print("Col Rpos LCM : ", end="")
    print(col_rpos_lcm)
    print("Row Rpos LCM : ", end="")
    print(row_rpos_lcm)
    print("")

    roots_row = root_lookup(row_rpos_lcm)
    roots_col = root_lookup(col_rpos_lcm)

    print("Row roots found :", end = " ")
    display_roots(roots_row)

    print("Col roots found :", end = " ")
    display_roots(roots_col)

    col_ccp_poly = root_to_coeff(roots_col)
    row_ccp_poly = root_to_coeff(roots_row)

    print("Col CCP poly : ", end="")
    print_poly(col_ccp_poly)
    print("Row CCP poly : ", end="")
    print_poly(row_ccp_poly)

    col_full = extend_columns(S_2t, col_ccp_poly)
    row_full = extend_rows(col_full, row_ccp_poly)
    print("")

    print("Full syndrome : ")
    print_grid(row_full)

    err_obtained = idffft_full(row_full)
    print("Error obtained : ")

    print_bin(err_obtained)

    if(err_obtained == r_in):
        print("pass")



c = [[0] * N for _ in range(N)]
c[1][9] = 1
c[2][9] = 1
c[1][10] = 1
c[2][10] = 1

ecc_2d(c)

Input : 
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Syndromes 1-2t : 
a^3  |  a^1  |  a^1  |  a^12  |  
a^8  |  a^6  |  a^6  |  a^2  |  
a^0  |  a^13  |  a^13  |  a^9  |  
a^3  |  a^1  |  a^1  |  a^12  |  

Column sigmas : 
['a^0', 'a^5', 'a^3']
['a^0', 'a^5', 'a^3']
['a^0', 'a^5', 'a^3']
['a^0', 'a^5', 'a^3']
Row sigmas : 
['a^0', 'a^13', 'a^4']
['a

In [56]:
#Full datapath (silent version)

def ecc_2d_silent(r_in):
    S_2t = dffft_2t(r_in)
    S_2t_arrnp = np.array(S_2t)
    S_2t_colwise = np.transpose(S_2t_arrnp).tolist()

    col_poly = []
    row_poly = []

    for j in S_2t:
        row_poly.append(BM_fixed(j))
    for j in S_2t_colwise:
        col_poly.append(BM_fixed(j))

    col_rpos = []
    row_rpos = []

    for sigma in col_poly:
        col_rpos.append(get_rpos(sigma))
    for sigma in row_poly:
        row_rpos.append(get_rpos(sigma))

    col_rpos_lcm = 0
    row_rpos_lcm = 0

    col_rpos_lcm = [0] * N
    row_rpos_lcm = [0] * N

    for rpos in col_rpos:
        col_rpos_lcm = [a | b for a, b in zip(col_rpos_lcm, rpos)]
    for rpos in row_rpos:
        row_rpos_lcm = [a | b for a, b in zip(row_rpos_lcm, rpos)]

    roots_row = root_lookup(row_rpos_lcm)
    roots_col = root_lookup(col_rpos_lcm)

    col_ccp_poly = root_to_coeff(roots_col)
    row_ccp_poly = root_to_coeff(roots_row)

    col_full = extend_columns(S_2t, col_ccp_poly)
    row_full = extend_rows(col_full, row_ccp_poly)

    err_obtained = idffft_full(row_full)

    if(err_obtained == r_in):
        print("Pass")

    return err_obtained == r_in



c = [[0] * N for _ in range(N)]
c[1][9] = 1
c[2][9] = 1
c[1][10] = 1
c[2][10] = 1

print(ecc_2d_silent(c))

Pass
True


In [ ]:
def unpack_to_grid(positions, n=N):
    grid = [[0] * n for _ in range(n)]
    for r, c in positions:
        grid[r][c] = 1
    return grid


def generate_all_patterns(n=N, t_val=t):
    """Generates distinct 2D quasi-cyclic error matrices up to t x t size."""
    seen = set()
    for rows in itertools.combinations(range(n), t_val):
        for cols in itertools.combinations(range(n), t_val):
            cells = [(r, c) for r in rows for c in cols]
            for mask in range(1, 1 << len(cells)):
                pos = tuple(
                    cells[k] for k in range(len(cells)) if (mask >> k) & 1
                )
                seen.add(pos)
    return list(seen)


# Run Limited Verification (First 4,000 cases)
patterns = generate_all_patterns(N, t)[:4000]
total_tests = len(patterns)

print(f"Testing first {total_tests:,} quasi-cyclic error patterns...\n")

passed_count = 0
failed_cases = []

for pos in tqdm(patterns, desc="Testing Patterns"):
    r_in = unpack_to_grid(pos, N)
    if ecc_2d_silent(r_in):
        passed_count += 1
    else:
        failed_cases.append(pos)
        break  # Stops immediately on failure

print("\n" + "=" * 45)
if passed_count == total_tests:
    print(
        f"RESULT: ALL PASSED! ({passed_count:,} / {total_tests:,} cases verified)"
    )
else:
    print(f"RESULT: FAILED at test #{len(failed_cases)}")
    print(f"Failing pattern coordinates: {failed_cases[0]}")
print("=" * 45)